# CassavaGenDup

CassavaGenDup is a Python/Jupyter Notebook tool for cassava varietal identification and duplicate detection using SNP genotyping data from the SNPY-Chip. The pipeline allows users to load genotype files, evaluate genotype quality, identify duplicated and unique samples, generate final reports, visualize genotype profiles, export VCF files, and perform kinship-based relationship analysis.

This notebook is designed to guide users step by step, so it can be used by researchers with limited programming experience.

## Requirements

Before running this notebook, make sure that Python and the required libraries are installed.

- Python 3.11 or higher
- Jupyter Notebook or JupyterLab

To install the required Python libraries, run:

pip install pandas numpy scipy scikit-learn matplotlib seaborn openpyxl

If you are using Anaconda or Miniconda, you can also create a dedicated environment for this tool before opening the notebook.

This pipeline expects the following folder structure:

data/: curated databases, SNP marker information, and optional sample lists\
input/: input genotype .csv files\
functions/: Python scripts used by the notebook\
results/: output files generated by the analysis

**Optional:** VCFtools is required only if the user wants to generate a `.relatedness2` file for kinship analysis. The notebook reads the `.relatedness2` file but does not run VCFtools directly.

## Load Pipeline Functions

In [ ]:
from functions.loading_chips import *
from functions.checking_chips import *
from functions.duplicated_analysis import *
from functions.final_table_result import *
from functions.heatmap_cluster import *
from functions.heatmap_samples import *
from functions.vcfFile_report import *
from functions.kinship_plot import *
from functions.merge_db import *
from functions.curate_db import *

## Reading Data

This section loads the genotype data files that will be analyzed by the CassavaGenDup pipeline.

The function loading_chips() reads all .csv files located in the input/ folder and converts them into a genotype matrix where rows correspond to samples and columns correspond to SNP markers. it can be used with any .csv file that contains, at minimum, the following information:

- A column with the genotype call for each sample-marker combination.
- A column with the SNP or assay identifier.
- A column with the sample name or sample identifier.

Before running the function, check that all input .csv files are placed inside the input/ folder.

Define the following parameters according to the column names in the input files:

final_call: name of the column containing the genotype call.\
assay_name: name of the column containing the SNP marker or assay ID.\
sample_name: name of the column containing the sample name.\
skiprows: number of rows to skip before the header line. If your .csv file already has the column names in the first row, use skiprows = 0.

For standard Fluidigm chip data, use:

final_call = "Final"\
assay_name = "Assay"\
sample_name = "Name"\
skiprows = 15

In [ ]:
final_call = "Final"
assay_name = "Assay"
sample_name  = "Name"
skiprows  = 15

In [ ]:
chip_all, gt_input_matrix, namesdb_input = loading_chips(final_call, assay_name, sample_name, skiprows)

## Checking Chips

This section evaluates the genotype quality of one or more chips loaded in the previous step.

The function checking_chips() calculates the genotype call frequency for each sample across the selected SNP markers and generates a heatmap to visualize the genotype calls.

Before running this section, make sure that:

1. The **Reading Data** section has already been run.
2. The file SNPY.csv is located in the data/ folder.
3. The column SNPY_PANEL in SNPY.csv indicates which SNP markers will be used in the analysis. Markers with SNPY_PANEL == "Y" will be included.

Define the following parameter:

chip_ids = the chip ID or chip IDs that you want to check. Use quotation marks and commas to separate multiple entries.

For example:

chip_ids = ['1382513272', '1382293072']

The function will generate:

A heatmap showing the genotype calls for each sample and SNP marker.\
A frequency table with the percentage of XX, XY, YY, and NoCall calls per sample.

The output files will be saved in the results/ folder.

Note: The chip ID is extracted from the input file name. For example, if the input file is 1382513272.csv, the chip ID will be 1382513272. This chip ID is also used to create a unique sample identifier by combining the sample name and the chip ID.


In [ ]:
chip_ids = ['1382513272'] 

In [ ]:
freq_all = checking_chips(gt_input_matrix, chip_ids)

## Detect Duplicates

This section identifies duplicated and unique samples using the selected SNP markers.

The function duplicated_analysis() compares the genotype profiles of the samples using Hamming distance and groups highly similar samples into clusters. Samples that do not group with any other sample are classified as unique.

Before running this section, make sure that:

1. The **Reading Data** section has already been run.
2. The file SNPY.csv is located in the data/ folder.
3. The file genotypedb_curated.csv is located in the data/ folder.
4. The column SNPY_PANEL in SNPY.csv indicates which SNP markers will be used in the analysis. Markers with SNPY_PANEL == "Y" will be included.

This function will:

- Combine the newly loaded samples with the curated genotype database.
- Filter samples according to the maximum allowed percentage of NoCall.
- Calculate the genetic distance between samples.
- Identify duplicated samples and unique samples.
- Assign duplicated samples to clusters.

No parameter needs to be modified before running this function.

**Note:** This step can take several minutes when analyzing large genotype databases.

In [ ]:
matrix, sq_dm, namesdb_tmp, freq_samples, filter_matrix = duplicated_analysis(gt_input_matrix, namesdb_input)

## Final Table

This section generates the final output table with the duplicate detection results.

The function final_table_result() combines the duplicate analysis results with genotype call frequencies and passport information.

Before running this section, make sure that:

1. The **Reading Data** section has already been run.
2. The **Detect Duplicates** section has already been run.
3. The file passportdb_curated.csv is located in the data/ folder.

Define the following parameter:

filename: name of the final Excel file that will be saved in the results/ folder.

The function will generate an Excel file containing:

- Sample identifiers.
- Sample names.
- Chip IDs.
- Duplicate cluster assignment.
- Number of samples per cluster.
- Genotype call frequencies.
- Passport information, when available.

The output file will be saved in the results/ folder.

**Note:** If no passport information is found for a sample, the table will indicate this in the passport_status column.

In [ ]:
filename = "Final_table.xlsx"
final_table = final_table_result(namesdb_tmp, matrix, freq_samples, filename)

## Plot Heatmap of Clusters

This section generates a genotype heatmap for one or more duplicate clusters.

The function heatmap_cluster() allows to visualize the genotype calls of the samples included in selected clusters. This is useful to inspect how similar the samples are within a duplicate group.

Before running this section, make sure that:

1. The **Detect Duplicates** section has already been run.
2. The file SNPY.csv is located in the data/ folder.
3. The cluster IDs that you want to plot are present in the duplicate analysis results.

Define the following parameters:

cluster_list: cluster ID or cluster IDs that you want to plot. Use quotation marks and commas to separate multiple entries.\
filename: name of the heatmap image file that will be saved in the results/ folder.\
dpi: image resolution. Higher values generate higher-quality images.\
figsize: figure size. Increase the second value if the heatmap contains many samples.

The function will generate:

A heatmap showing the genotype calls for the selected cluster or clusters. The output image will be saved in the results/ folder.

In [ ]:
cluster_list = ['1212']
filename = "Cluster_heatmap.png"
dpi = 300

In [ ]:
heatmap_cluster(cluster_list, matrix, filename, dpi)

## Plot Heatmap of Samples

This section generates a genotype heatmap for a user-defined list of samples.

The function heatmap_samples() allows to visualize the genotype calls of specific samples selected from the duplicate analysis results.

Before running this section, make sure that:

1. The **Detect Duplicates** section has already been run.
2. The file SNPY.csv is located in the data/ folder.
3. The sample list file is located in the data/ folder.
4. The sample list file contains one sample ID per row and does not have a header.

Define the following parameters:

sample_list_file: name of the .csv file containing the list of samples to plot. This file must contain one sample ID per row and must not have a header.\
filename: name of the heatmap image file that will be saved in the results/ folder.\
dpi: image resolution. Higher values generate higher-quality images.\
long: figure height. Increase this value if the heatmap contains many samples.

The function will generate:

A heatmap showing the genotype calls for the selected samples. The output image will be saved in the results/ folder.

**Note:** The sample IDs in the sample list file must match the unique sample identifiers generated by the pipeline. These identifiers are created by combining the sample name and the chip ID.

In [ ]:
sample_list_file = 'samples.csv'
filename = "samples_heatmap.png"
dpi = 300
long = 10

In [ ]:
heatmap_samples(sample_list_file, matrix, filename, dpi, long)

## VCF Report

This section generates a VCF file for a user-defined list of samples.

The function vcfFile_report() converts the selected genotype data into VCF format. This file can be used for downstream analyses, such as relatedness analysis with external tools.

Before running this section, make sure that:

1. The **Detect Duplicates** section has already been run.
2. The file SNPY.csv is located in the data/ folder.
3. The sample list file is located in the data/ folder.

Define the following parameters:

sample_list_file: name of the .csv file containing the list of samples to include in the VCF file. This file must contain one sample ID per row and must not have a header.\
output_vcf: name of the VCF file that will be saved in the results/ folder.

The function will generate:

A VCF file containing the selected samples. The output file will be saved in the results/ folder.

**Note:** The sample IDs in the sample list file must match the unique sample identifiers generated by the pipeline. These identifiers are created by combining the sample name and the chip ID.

In [ ]:
sample_list_file = 'samples.csv'
output_VCF = "samples.vcf"

In [ ]:
vcfFile_report(sample_list_file, matrix, output_VCF)

## Kinship


This section generates a kinship heatmap and a pairwise relationship table using a .relatedness2 file.

The function kinship_plot() reads the kinship coefficients generated by VCFtools and classifies the relationship between samples based on the PHI value.

Before running this section, make sure that:

1. The **Detect Duplicates** section has already been run.
2. A .relatedness2 file has already been generated using VCFtools.
3. The .relatedness2 file is located in the data/ folder.
4. The sample IDs in the .relatedness2 file match the unique sample identifiers generated by the pipeline.

Define the following parameters:

kinship_file: name of the .relatedness2 file located in the data/ folder.\
kinship_output: name of the output .csv file that will be saved in the results/ folder.\
dpi: image resolution for the kinship heatmap. Higher values generate higher-quality images.

The function will generate:

- A kinship heatmap showing the relationship among samples.
- A pairwise relationship table with the PHI value and relationship category for each pair of samples.

The output files will be saved in the results/ folder.

**Note:** This function does not run VCFtools. It uses the .relatedness2 file generated externally from a VCF file. Self-comparisons between the same sample are excluded from the final pairwise table.

In [ ]:
kinship_file = 'samples.relatedness2'
kinship_output = 'samples_Kinship.csv'

In [ ]:
pairwise_labels = kinship_plot(kinship_file, kinship_output, matrix)

# Merge and curate


This section is intended for genotype database maintenance. It allows to add newly loaded genotype data to an existing curated database or remove selected samples from a database.

These steps should be used carefully and only when the goal is to update or clean the genotype database used by the pipeline. The new database files generated in this section will be saved in the results/ folder.

**Note:** Always review the output files before replacing the original database files in the `data/` folder.

### Merge

The function merge_db() combines the genotype data loaded in the Reading Data section with an existing database file. This is useful to update the genotype database by adding new samples.

Before running this section, make sure that:

1. The **Reading Data** section has already been run.
2. The existing genotype database file is located in the data/ folder.
3. The newly loaded samples are not already present in the existing database.

Define the following parameters:

input: name of the existing genotype database file located in the data/ folder.\
output: name of the new merged database file that will be saved in the results/ folder.

The function will generate:

A new genotype database containing both the newly loaded samples and the existing database samples. The output file will be saved in the results/ folder.

**Note:** This function checks whether any newly loaded sample already exists in the database. If duplicated sample IDs are detected, the function will stop and report the duplicated samples.

In [ ]:
input = "genotypedb.csv"
output = "genotypedb_V2.csv"

In [ ]:
merged = merge_db(gt_input_matrix, namesdb_input, input, output)

### Curate

The function curate_db() uses a user-defined sample list to remove specific samples from an existing genotype database. This is useful to clean the database by removing duplicated, incorrect, or unwanted samples.

Before running this section, make sure that:

1. The genotype database file is located in the data/ folder.
2. The sample list file is located in the data/ folder.
3. The sample list file contains one sample ID per row and does not have a header.
4. The sample IDs in the list match the unique sample identifiers used in the database.

Define the following parameters:

delete_file: name of the .csv file containing the list of samples to remove. This file must contain one sample ID per row and must not have a header.\
input_db: name of the genotype database file that will be curated. This file must be located in the data/ folder.\
output_db: name of the curated database file that will be saved in the results/ folder.

The function will generate:

A new curated genotype database without the selected samples.The output file will be saved in the results/ folder.

**Note:** This function does not overwrite the original database. It creates a new curated file in the results/ folder. Always review the output file before replacing any database file in the data/ folder.

In [ ]:
delete = "samples_to_delete.csv"
input = "genotypedb.csv"
output = "genotypedb_V3.csv"

In [ ]:
curated = curate_db(delete, input, output)

## Citation

If you use CassavaGenDup or the SNPY-Chip data structure in your research, please cite the associated manuscript: